Exercise 1 — Create an Orders DataFrame

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType(
    [
        StructField("order_id", IntegerType(), False),
        StructField("customer_name", StringType(), False),
        StructField("product", StringType(), False),
        StructField("amount", IntegerType(), False),
        StructField("status", StringType(), False)
    ]
)

raw_data = [
    (1001, "Alice Smith", "Laptop", 1200, "Shipped"),
    (1002, "Bob Jones", "Mouse", 25, "Delivered"),
    (1003, "Charlie Brown", "Monitor", 250, "Pending"),
    (1004, "Diana Prince", "Keyboard", 75, "Shipped"),
    (1005, "Ethan Hunt", "Headphones", 120, "Delivered"),
    (1006, "Fiona Gallagher", "USB Cable", 15, "Cancelled"),
    (1007, "George Clark", "Office Chair", 185, "Delivered"),
    (1008, "Hannah Abbott", "Webcam", 60, "Shipped"),
    (1009, "Ian Malcolm", "External SSD", 130, "Pending"),
    (1010, "Julia Roberts", "Desk Mat", 25, "Delivered"),
    (1011, "Kevin Hart", "Monitor", 300, "Shipped"),
    (1012, "Laura Croft", "Laptop", 1450, "Pending"),
    (1013, "Michael Scott", "Office Chair", 210, "Delivered"),
    (1014, "Pam Beesly", "Keyboard", 85, "Cancelled"),
    (1015, "Dwight Schrute", "Headphones", 95, "Shipped")
]

df = spark.createDataFrame(raw_data, schema)

display(df)
df.printSchema()
df.count()

Exercise 2 — Transform the Orders DataFrame

In [0]:
from pyspark.sql import functions as F

result = df.select("order_id", "product", "amount", "status").filter(
    (F.col("amount") > 200)
    & (F.col("status") == 'Delivered')
    )

result = (
    result
    .withColumn("tax", F.round(F.col("amount") * 0.1, 2))
    .orderBy(F.col("amount").desc())
)

display(result)

Exercise 3 — Save and Query Your Data

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("lab.dev.practice_orders")
)

In [0]:
saved_df = spark.table("lab.dev.practice_orders")

sql_saved_df = spark.sql('''
                         SELECT * FROM lab.dev.practice_orders
                         ''')

display(sql_saved_df)

Exercise 4 — Practice MERGE INTO

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType(
    [
        StructField("order_id", IntegerType(), False),
        StructField("customer_name", StringType(), False),
        StructField("product", StringType(), False),
        StructField("amount", IntegerType(), False),
        StructField("status", StringType(), False)
    ]
)

raw_updates_data = [
    (1003, "Charlie Brown", "Monitor", 250, "Completed"),
    
    (1012, "Laura Croft", "Laptop", 1500, "Pending"),
    
    (1016, "Jim Halpert", "Webcam", 70, "Pending")
]

df_updates = spark.createDataFrame(raw_updates_data, schema)

display(df_updates)
df_updates.printSchema()

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "lab.dev.practice_orders")

delta_table.alias("target").merge(
    df_updates.alias("source"),
    "target.order_id = source.order_id"
).whenMatchedUpdate(set={
    "status": "source.status",
    "amount": "source.amount"
}).whenNotMatchedInsert(values={
    "order_id": "source.order_id",
    "customer_name": "source.customer_name",
    "product": "source.product",
    "amount": "source.amount",
    "status": "source.status"
}).execute()

In [0]:
df = spark.table("lab.dev.practice_orders")

display(df)